In [114]:
import pandas as pd
import numpy as np

In [115]:
episode_df = pd.read_csv(r"dataset\fineTuned\top10_episodes.csv",index_col=0)
embeddings = np.load(r"dataset\notFineTuned\embeddings\balanced_episode_embeddings.npy")
episode_df['published_date'] = pd.to_datetime(episode_df['published_date'])
episode_df = episode_df.reset_index()

In [116]:
print("Number of sentences:", len(episode_df))
episode_df.head()

Number of sentences: 337


,orig_index,content_id,sentence,published_date,category,timestamp,episode,t
0,0,1442420,"Until now, the AirPods Pro were all about keep...",2024-09-17,Gadgets,2024-09-17,18,208
1,1,1442420,Apple says the latest AirPods Pro 2 can be use...,2024-09-17,Gadgets,2024-09-17,18,208
2,3,1442420,The company is also planning to integrate a he...,2024-09-17,Gadgets,2024-09-17,18,208
3,5,1442420,At Apple's annual September product launch eve...,2024-09-17,Gadgets,2024-09-17,18,208
4,7,1442420,The new AirPods also have a slightly modified ...,2024-09-17,Gadgets,2024-09-17,18,208


In [117]:
indices = episode_df["orig_index"].values
episode_embeddings = embeddings[indices]

## Clustering within episode to find events

In [118]:
import hdbscan

In [119]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=4,
    metric='euclidean',
    prediction_data=True
) 

event_labels = clusterer.fit_predict(episode_embeddings)

c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [120]:
episode_df["event_id"] = event_labels

In [121]:
print("Unique event IDs:", set(event_labels))
print("Number of clusters (excluding -1):", len(set(event_labels)) - (1 if -1 in event_labels else 0))
print("Noise points:", list(event_labels).count(-1))

Unique event IDs: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, -1}
Number of clusters (excluding -1): 10
Noise points: 31


In [122]:
episode_df = episode_df[episode_df["event_id"] != -1].copy()

In [123]:
event_counts = episode_df["event_id"].value_counts()

print(event_counts)

4    95
0    35
8    35
6    31
2    25
1    21
3    20
7    15
9    15
5    14
Name: event_id, dtype: int64


In [124]:
valid_events = event_counts[event_counts >= 3].index

episode_df = episode_df[episode_df["event_id"].isin(valid_events)].copy()
episode_df.shape

(306, 9)

In [125]:
episode_df = episode_df.sort_values(by=["event_id", "published_date"])

In [126]:
for event in episode_df["event_id"].unique():
    print("\n======================")
    print(f"Event ID: {event}")
    print("======================")
    
    samples = episode_df[episode_df["event_id"] == event]["sentence"].head(5)
    
    for s in samples:
        print("-", s)


Event ID: 0
- Children are developing nearsightedness earlier in life, and while looking too long and too close at phones and tablets may play a role, the answer is not that simple, according to a Connecticut paediatric ophthalmologist in the US.
- “We are seeing an increased rate of myopia over the last, I would say, 20 years or so, and it is projected to increase,” said Dr Majida Gaffar, division head of ophthalmology at Connecticut Children’s Medical Center.
- “It’s definitely something that ophthalmologists and optometrists are looking at to slow the Myopia affects about 5% of preschoolers, 9% of school-aged children and 30% of teens, according to While there is no cause for myopia, the formal name for nearsightedness, there are correlations, such as “near work and diet ... environmental factors,” Gaffar said.
- Also, the child of someone with myopia is more likely to be nearsighted.
- “I’ll always tell my patients to limit as much as possible, even though they haven’t really foun

## Pair-wise dataset preparation for fine-tuning

In [127]:
grouped = episode_df.groupby("event_id")

In [128]:
from itertools import combinations
import random

### Positive pair (Label:1)

In [129]:
positive_pairs = []

for event_id, group in grouped:
    sentences = group["sentence"].tolist()
    
    pairs = list(combinations(sentences, 2))
    pairs = random.sample(pairs, min(len(pairs), 70)) 
    
    for s1, s2 in pairs:
        positive_pairs.append((s1, s2, 1))

print(len(positive_pairs))

700


In [130]:
for i in range(3):
    pair_id = random.randint(0,len(positive_pairs))
    print("\n======================")
    print(f"Pair {pair_id}:")
    print("======================")
    print(positive_pairs[pair_id][0])
    print(positive_pairs[pair_id][1])


Pair 514:
Like its direct predecessor, the M5 iPad Pro has a thin design and includes a multilayered "tandem” OLED display - an approach that offers more brightness and efficiency
Since its release in early 2024, the Vision Pro has been viewed as too heavy and expensive

Pair 94:
I, for one, took this opportunity to disconnect my Roku TV from the Internet and plug in a different streaming device with less onerous terms, an old Apple TV.
But Roku is a bigger offender, as it collects much more information than it needs to provide a device that runs streaming apps, including information about your employment, education and religious beliefs, she said.

Pair 291:
The role played by 5G in this scenario comes largely in two areas.
He says with faster speeds and near-instant connectivity, 5G can support complex technologies like real-time holographic imaging.


### Negative pairs (Label:0)

In [131]:
negative_pairs = []

event_ids = list(grouped.groups.keys())

num_neg = len(positive_pairs)

for _ in range(num_neg):
    e1, e2 = random.sample(event_ids, 2)
    
    s1 = random.choice(grouped.get_group(e1)["sentence"].tolist())
    s2 = random.choice(grouped.get_group(e2)["sentence"].tolist())
    
    negative_pairs.append((s1, s2, 0))

print(len(negative_pairs))

700


In [135]:
for i in range(3):
    pair_id = random.randint(0,len(negative_pairs))
    print("\n======================")
    print(f"Pair {pair_id}:")
    print("======================")
    print(negative_pairs[pair_id][0])
    print(negative_pairs[pair_id][1])


Pair 637:
So he came up with a workaround to disconnect his Roku TV from the Internet and use it as a normal TV without Roku’s apps, which include Netflix, Hulu and other streaming services.
The company is still working on new AirPods hardware, including a third-generation version of the AirPods Pro.

Pair 627:
Supporters of the bill expressed their concern that neural data could be used to decode a person’s thoughts and feelings or to learn sensitive facts about an individual’s mental health, such as whether someone has epilepsy.
He also took the opportunity to showcase a fully functioning keyboard he crafted from Lego bricks.

Pair 177:
“The longer the eye, the more likely it is to have myopia,” she said.
In addition, Singapore Airlines and Thai Airways both prohibit the use or charging of portable power banks at all during flights.


### Combining the positive and negative pair

In [133]:
all_pairs = positive_pairs + negative_pairs

pairs_df = pd.DataFrame(all_pairs, columns=["sentence1", "sentence2", "label"])
pairs_df = pairs_df.sample(frac=1).reset_index(drop=True)

print(pairs_df.shape)
pairs_df.head()

(1400, 3)


,sentence1,sentence2,label
0,While Southwest is the first US airline to res...,"Until now, the AirPods Pro were all about keep...",0
1,The report analysed 30 consumer neurotechnolog...,While some eye doctors will give young childre...,0
2,With many consumers unwilling to splurge on a ...,"Mark Andrews, waste and recycling fires lead f...",0
3,The iPad Pro also gains the company’s in-house...,That company’s just-released Meta Ray-Ban Disp...,1
4,"If the bill passes, its legal implications wou...","For Bryan Lin, the advancement of technology b...",0


In [134]:
pairs_df.to_csv(r"dataset\fineTuned\pairwise_dataset.csv", index=False)